En este notebook entreno un modelo propio con scikit-learn, lo evalúo y lo exporto con joblib para usarlo después en una API con FastAPI.
Uso el dataset Iris que viene incluido en scikit-learn. Tiene 150 flores de 3 especies. Las features (entrada) son 4 medidas en cm: largo y ancho del sépalo y del pétalo. El target(objetivo) es la especie. Acá cargo los datos y miro sus columnas y su tamaño.

In [1]:
# Carga el dataset y separa features (X) y target (y), para documentar de dónde salen los datos.
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris(as_frame=True)
df = iris.frame
X = df[iris.feature_names]
y = df["target"]
print(iris.feature_names, iris.target_names, df.shape)


['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'] ['setosa' 'versicolor' 'virginica'] (150, 5)


División y entrenamiento
Separo 80% de los datos para entrenar y 20% para probar, así evalúo el modelo con datos que nunca vio. Uso `stratify=y` para que las 3 especies estén repartidas parejo en ambos conjuntos y `random_state=42` para que el resultado se pueda repetir. El modelo es un Random Forest: un conjunto de árboles de decisión que votan la clase.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (120, 4) Test: (30, 4)


Evaluación

Evalúo con accuracy porque las 3 clases están balanceadas, y con el reporte por clase (precision, recall y f1) para ver si alguna especie se confunde más que otra.

In [3]:
from sklearn.metrics import accuracy_score, classification_report

pred = modelo.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names))

Accuracy: 0.9
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



Interpretación de los resultados

El modelo obtuvo un accuracy de 0.90: acertó 27 de las 30 flores de prueba. Setosa se clasificó perfecto (precision y recall de 1.00), porque es la especie más separable de las otras dos. Los 3 errores ocurrieron entre versicolor y virginica: del reporte se deduce que 2 virginica se clasificaron como versicolor y 1 versicolor como virginica. Como el conjunto de prueba es chico, un solo error cambia el accuracy en 3.3 puntos, así que conviene no sobreinterpretar el 0.90.

## 4. Exportación del modelo
Guardo el modelo entrenado con joblib en el archivo `modelo.joblib`. Registro la versión de scikit-learn porque la API tiene que usar la misma para poder cargarlo sin errores, y el peso del archivo en MB porque las plataformas gratuitas tienen límites.

In [4]:
# Exporta el modelo con joblib y registra versión de scikit-learn y peso en MB, como pide la consigna.
import joblib, os, sklearn

joblib.dump(modelo, "modelo.joblib")
print("scikit-learn:", sklearn.__version__)
print("Peso (MB):", os.path.getsize("modelo.joblib") / 1e6)

scikit-learn: 1.9.1
Peso (MB): 0.168049
